# Data Analytics Project 1 — Data Cleaning & Preparation

**Dataset:** Dataset for Data Analytics.xlsx

## Objective
Clean and prepare the raw e-commerce order dataset by:
- identifying missing/null values,
- removing duplicate records,
- correcting data types and formats for dates, numbers, and text,
- validating the final dataset.

This notebook is designed to demonstrate the data-cleaning workflow required for the project.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 2. Load the Raw Dataset

In [ ]:
# Put the Excel file in the same folder as this notebook
file_path = "Dataset for Data Analytics.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()


## 3. Initial Data Inspection

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nSummary:")
display(df.describe(include="all").T)


## 4. Check Missing Values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

print("Columns containing missing values:")
display(missing)

print("\nTotal missing cells:", int(df.isna().sum().sum()))


### Missing-value treatment

The raw dataset contains missing values in `CouponCode`. For this project, a missing coupon code is interpreted as the order having **no coupon**, so it is replaced with `No Coupon`.

This assumption should be mentioned in the project documentation rather than silently treating the missing value as an error.


## 5. Check for Duplicate Records

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate OrderIDs:", df["OrderID"].duplicated().sum())

duplicate_ids = df[df["OrderID"].duplicated(keep=False)].sort_values("OrderID")
display(duplicate_ids)


## 6. Standardize Column Names and Text

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    df[col] = df[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print(df.columns.tolist())


## 7. Correct Data Types

In [ ]:
# Date
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Integer fields
for col in ["Quantity", "ItemsInCart"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# Numeric/currency fields
for col in ["UnitPrice", "TotalPrice"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)


## 8. Handle Missing Values

In [ ]:
df["CouponCode"] = df["CouponCode"].fillna("No Coupon")

print("Missing values after treatment:")
display(df.isna().sum())


## 9. Remove Duplicates

In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates()

# OrderID should uniquely identify an order.
df = df.drop_duplicates(subset="OrderID", keep="first")

print("Rows after duplicate removal:", len(df))
print("Duplicate rows remaining:", df.duplicated().sum())
print("Duplicate OrderIDs remaining:", df["OrderID"].duplicated().sum())


## 10. Validate Dates, Numbers and Text

In [ ]:
# Final text cleanup
for col in [
    "OrderID", "CustomerID", "Product", "ShippingAddress",
    "PaymentMethod", "OrderStatus", "TrackingNumber",
    "CouponCode", "ReferralSource"
]:
    df[col] = df[col].astype("string").str.strip()

# Currency precision
df["UnitPrice"] = df["UnitPrice"].round(2)
df["TotalPrice"] = df["TotalPrice"].round(2)

print("Invalid/missing dates:", df["Date"].isna().sum())
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate OrderIDs:", df["OrderID"].duplicated().sum())


## 11. Validate TotalPrice

In [ ]:
expected_total = df["Quantity"].astype(float) * df["UnitPrice"]

price_errors = (df["TotalPrice"] - expected_total).abs() > 0.01

print("Rows with TotalPrice calculation mismatch:", price_errors.sum())


## 12. Final Clean Dataset

In [ ]:
print("Final shape:", df.shape)
display(df.head(10))
display(df.dtypes)


## 13. Export the Cleaned Dataset

In [ ]:
df.to_csv("cleaned_dataset.csv", index=False)

with pd.ExcelWriter("cleaned_dataset.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Cleaned_Data", index=False)

print("Cleaned CSV and Excel files created successfully.")


## 14. Final Quality Checklist

The cleaned dataset should satisfy these conditions:

- Missing values are handled.
- Exact duplicate rows are removed.
- `OrderID` contains no duplicates.
- `Date` is stored as a proper datetime value.
- Quantity and cart-count columns are numeric.
- Price columns are numeric and rounded to two decimal places.
- Text fields are stripped of unnecessary whitespace.
- `TotalPrice` agrees with `Quantity × UnitPrice` within a 0.01 tolerance.

This completes the Data Cleaning & Preparation stage.
